In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import build_dataset
from src.misc import load_config
from src.predict import AnchorSlice, load_predict_config, load_predictor


def select_anchor_image(dataset, axis):
    for index in torch.randperm(len(dataset)).tolist():
        sample = dataset[index]
        if int(sample[2]) == axis:
            return sample[0][0].numpy().astype("uint8")
    raise ValueError(f"dataset contains no samples for axis {axis}.")

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
ANCHOR_AXIS = 0
AXIS_NAMES = ("axis 0", "axis 1", "axis 2")
NEIGHBOR_OFFSETS = (-2, -1, 0, 1, 2)

In [ ]:
config = load_predict_config(PREDICT_CONFIG)
run_dir = ROOT / config.run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = load_predictor(run_dir, device=device)
options = config.make_options(predictor)

data_config = Namespace(**load_config(run_dir / "model.yaml"))
data_config.data_dir = {
    axis: (path if (path := Path(value)).is_absolute() else run_dir / path).resolve()
    for axis, value in data_config.data_dir.items()
}
data_config.augment = False
dataset = build_dataset(data_config)
anchor_image = select_anchor_image(dataset, ANCHOR_AXIS)
anchor = AnchorSlice(
    anchor_image,
    axis=ANCHOR_AXIS,
    index=options.volume_size // 2,
)

In [ ]:
start = time.perf_counter()
volume, stats = predictor.predict(options, anchors=[anchor])
elapsed = time.perf_counter() - start
fractions = [round(value, 4) for value in stats["phase_fractions"]]
print(
    f"volume={tuple(volume.shape)} anchor={anchor.index} "
    f"elapsed={elapsed:.1f}s fractions={fractions}"
)

In [ ]:
center = options.volume_size // 2
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for axis_id, axis in enumerate(axes):
    axis.imshow(
        volume.select(axis_id, center),
        cmap="gray",
        vmin=0,
        vmax=options.num_phases - 1,
        interpolation="nearest",
    )
    axis.set_title(f"{AXIS_NAMES[axis_id]} / CENTER", fontsize=16, fontweight="bold")
    axis.axis("off")
fig.suptitle("Orthogonal center slices", fontsize=20, fontweight="bold")
fig.subplots_adjust(top=0.82, wspace=0.08)
plt.show()

In [ ]:
neighbors = [
    volume.select(anchor.axis, anchor.index + offset).numpy()
    for offset in NEIGHBOR_OFFSETS
]
panels = [(anchor.image, "ANCHOR INPUT", True)] + [
    (
        image,
        "GENERATED AT ANCHOR" if offset == 0 else f"NEIGHBOR {offset:+d}",
        offset == 0,
    )
    for offset, image in zip(NEIGHBOR_OFFSETS, neighbors)
]

fig, axes = plt.subplots(2, 3, figsize=(11, 8))
for axis, (image, title, highlighted) in zip(axes.ravel(), panels):
    axis.imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=options.num_phases - 1,
        interpolation="nearest",
    )
    axis.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        color="crimson" if highlighted else "black",
    )
    axis.axis("off")
fig.suptitle(
    f"Anchor neighborhood / {AXIS_NAMES[anchor.axis]} / index {anchor.index}",
    fontsize=20,
    fontweight="bold",
)
fig.subplots_adjust(top=0.88, hspace=0.25, wspace=0.08)
plt.show()

In [ ]:
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(volume.numpy(), name="MPDD volume")
viewer